# ColGraphRAG WebQA — 단계별 파이프라인 튜토리얼

이 노트북은 **질의 기반 멀티모달 GraphRAG** 파이프라인을 따라갑니다.

| 단계 | 스크립트 | 역할 |
|-------|--------|------|
| 0 | `export_webqa_slice.py` *또는* 미리 만든 슬라이스 | JSONL 코퍼스 슬라이스 생성·복사 |
| 2 | `pattern.py` | 질문별 그래프 패턴 (LLM) |
| 3 | `extraction.py` | 엔티티·관계 추출 (LLM) |
| 4 | `construct.py` | NetworkX → GraphML |
| 5 | `inference.py` | ColEmbed MaxSim 검색 + Gemma 답변 |
| 6 | `eval/evaluate_webqa_qa.py` | QA-FL / QA-Acc / QA 지표 |

**`webqa_slice`의 slice:** WebQA **질문·텍스트·이미지**(및 선택적 테이블) JSONL을 **이번 run**에서 파이프라인이 찾을 수 있도록 `result/<RUN_ID>/webqa_slice/` 아래 **고정 파일명**(`webqa_questions.jsonl`, `webqa_texts.jsonl`, `webqa_images.jsonl`, …)으로 묶어 둔 **실행용 데이터 번들**을 뜻합니다. NumPy/PyTorch tensor의 차원 **slice**와는 무관합니다. 이 노트북의 수동 절에서는 보통 `data/webqa/.../webqa_slice` 토이 묶음을 그대로 복사하고, 더 큰 코퍼스에서는 `export_webqa_slice.py`로 같은 식의 폴더를 준비합니다. **`N_QUERIES`** 같은 값은 slice 정의가 아니라 이후 단계에서 **처리할 질문 수만 제한**하는 설정입니다.

**2단계 패턴:** 질문 하나마다 LLM이 **엔티티·관계 타입 골격**(부분 그래프 설계도)을 만들고, 3단계 **추출**이 그 안에 텍스트·이미지 증거를 채웁니다. pattern cache는 **정답이 아니라** 추출·그래프 구축을 위한 **질의 주도 스키마**에 가깝습니다. (세부 설명은 아래 **패턴 (`pattern.py`)** 절.)

**사전 준비:** 아래 **환경 설정**(venv + `pip install -r requirements.txt`)을 끝내세요. 실제 LLM·ColEmbed 실행 전에는 `util/download_models.py`로 **Hugging Face 가중치**를 받아야 합니다(아래 **Hugging Face 모델 다운로드** 참고). **Ollama** 가중치는 선택적으로 아래 **Ollama 모델 다운로드** 절에서 받을 수 있습니다. 실제 Gemma를 쓰려면 CUDA GPU가 필요합니다. 모델 없이 빠르게 연결만 확인하려면 파이프라인 셀에서 `--dry-run`을 사용하세요.

**작업 디렉터리:** Jupyter를 저장소 루트 `colgraphrag_webqa`에서 실행하거나, **저장소 루트 경로 잡기** 셀에서 `REPO`를 맞춥니다.

**두 가지 방식:** **한 번에 실행(End-to-end)** 으로 통째로 돌리거나, **단계별 수동(Step-by-step)** 으로 단계마다 실행합니다. 같은 세션에서 둘 다 쓰지 마세요(서로 다른 `RUN_ID`를 쓰거나 출력을 이해하는 경우만 예외).


## 가상환경과 CUDA 의존성

노트북을 열기 **전에** 로컬에서 **한 번만** 설정합니다. 명령은 저장소 루트 `colgraphrag_webqa/`(`requirements.txt`, `inference.py`, `notebook/`이 있는 폴더)에서 실행한다고 가정합니다.

### 가상환경(venv) 만들기

**Linux / macOS**

```bash
cd /path/to/colgraphrag_webqa
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip setuptools wheel
```

**Windows (PowerShell)**

```powershell
cd C:\path\to\colgraphrag_webqa
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -U pip setuptools wheel
```

**Python 3.10 이상**을 쓰세요(3.11 권장).

### 노트북을 `.venv`에 연결하기 (VS Code / Cursor)

venv를 만든 뒤, 노트북 셀 실행에 사용할 파이썬을 고릅니다.

1. 이 `.ipynb` 파일을 엽니다.
2. 노트북 우측 상단에서 **커널 선택**(또는 명령 팔레트의 **Notebook: Select Notebook Kernel**).
3. **Python 환경**을 선택합니다(먼저 **다른 커널 선택…**이 필요할 수 있음).
4. **기존 Python 환경**에서 저장소 안 인터프리터를 고릅니다:  
   **`.venv/bin/python`** (Linux/macOS) 또는 **`.venv\Scripts\python.exe`** (Windows).  
   목록에 없으면 **인터프리터 경로 입력…**으로 직접 지정합니다.

나중에 아래 **Jupyter 커널**로 표시 이름을 등록하면 `.venv` 경로를 찾지 않고 그 커널만 선택해도 됩니다.

### venv 안에 CUDA용 PyTorch 명시 설치

venv를 **활성화**하고 `pip`를 올린 뒤, 먼저 CUDA 12.4 휠을 설치합니다(`requirements.txt`와 동일한 고정):

```bash
pip install torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124 \
  --extra-index-url https://pypi.org/simple
```

그다음 나머지 의존성(networkx, transformers 등)을 설치합니다:

```bash
pip install -r requirements.txt
```

버전이 맞으면 `pip`가 이미 설치한 CUDA PyTorch를 유지합니다. 그렇지 않으면 아래 **한 번에 설치** 방식을 권장합니다.

### `requirements.txt`로 GPU 스택 한 번에 설치

파일 상단에 `--index-url https://download.pytorch.org/whl/cu124`로 **CUDA 12.4용 PyTorch**가 고정되어 있습니다. 전체를 한 번에 설치하면 위 두 단계와 동일합니다:

```bash
pip install -r requirements.txt
```

설치 후 PyTorch가 CUDA를 보는지 확인합니다:

```bash
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

NVIDIA 드라이버와 GPU가 있으면 빌드 이름에 `+cu124`가 보이고 `True`가 나와야 합니다.

### Jupyter 커널(선택)

이 노트북을 venv 안에서 돌리려면:

```bash
pip install ipykernel
python -m ipykernel install --user --name colgraphrag-webqa --display-name "Python (colgraphrag_webqa)"
```

VS Code / Cursor / Jupyter에서 **Python (colgraphrag-webqa)** 커널을 선택합니다.

**CPU 전용 참고:** `requirements.txt`는 CUDA 기준입니다. CPU만으로 스모크 테스트하려면 별도 고정이 필요합니다(여기서는 다루지 않음). 아래 파이프라인은 Gemma를 GPU에 올리지 않고 `--dry-run`을 지원합니다.

---


## 사용 가능한 GPU

venv를 활성화하고 PyTorch(`requirements.txt`) 설치를 마친 뒤 실행하세요. 가능하면 `nvidia-smi`로 **NVIDIA 드라이버/GPU**를 보여 주고, 이어서 `torch`로 **PyTorch CUDA** 상태를 출력합니다(torch가 아직 없으면 torch 관련 줄은 건너뜁니다).

In [2]:
# nvidia-smi로 GPU와 드라이버를 확인하고, 같은 커널에서 torch CUDA 사용 가능 여부를 출력합니다.

import shutil
import subprocess

nv = shutil.which("nvidia-smi")
if nv:
    r = subprocess.run(
        [
            nv,
            "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=15,
    )
    if r.returncode == 0:
        print("nvidia-smi (GPU 목록):\n")
        print(r.stdout.strip() or "(비어 있음)")
    else:
        print("nvidia-smi 실패:", r.stderr)
else:
    print("PATH에 nvidia-smi 없음 — NVIDIA CLI가 없거나 GPU 머신이 아님.")

print()

try:
    import torch
except ImportError:
    print("torch: 아직 설치되지 않음 — 먼저 pip install -r requirements.txt를 완료하세요.")
else:
    print("torch:", torch.__version__)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print("torch.cuda.device_count():", n)
        for i in range(n):
            print(f"  [{i}]", torch.cuda.get_device_name(i))
            print("      할당된 메모리 MB:", torch.cuda.memory_allocated(i) / 1e6)

nvidia-smi (GPU 목록):

0, NVIDIA RTX A6000, 570.195.03, 49140 MiB, 4 MiB, 48546 MiB

torch: 2.6.0+cu124
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
  [0] NVIDIA RTX A6000
      할당된 메모리 MB: 0.0


## 저장소 루트와 경로 맞추기

파이프라인은 `PYTHONPATH`에 저장소 루트가 들어가야 `mllm`, `util`, `prompt` 같은 import가 해석됩니다.

In [3]:
# inference.py가 있는 폴더를 저장소 루트로 삼아 cwd·sys.path를 맞춥니다.

import os
import sys
from pathlib import Path

# cwd 또는 상위 디렉터리에서 inference.py가 있는 저장소 루트를 찾음
_here = Path.cwd()
REPO = None
for _p in [_here, *list(_here.resolve().parents)]:
    if (_p / "inference.py").is_file():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError(
        "저장소 루트(inference.py가 있는 폴더)를 찾을 수 없습니다. "
        "Jupyter cwd를 colgraphrag 루트로 맞추거나 해당 폴더로 cd한 뒤 이 셀을 다시 실행하세요."
    )

assert (REPO / "inference.py").is_file(), f"REPO를 저장소 루트로 맞추세요; 현재 {REPO}"

os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("REPO =", REPO.resolve())
print("cwd =", Path.cwd())

REPO = /workspace/late-interaction-mm-graph-rag
cwd = /workspace/late-interaction-mm-graph-rag


## 환경 확인(선택)

**실제** 추론에서는 일반적인 단일 GPU 환경에서 VRAM을 줄이려고 `GEMMA4_E4B_IT_TORCH_DTYPE=bf16`을 쓰는 것이 좋습니다.

In [4]:
# 이 커널에서 torch 버전과 CUDA 사용 가능 여부만 간단히 다시 출력합니다.

import torch

print("torch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("디바이스:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu124
CUDA 사용 가능: True
디바이스: NVIDIA RTX A6000


## 설정 파일

- `config/data.yaml` — 데이터셋 경로(`WEBQA_DATA_ROOT`, 슬라이스 위치).
- `config/model.yaml` — Gemma, ColEmbed, fluency 모델 경로 및 HF 저장소 ID.
- 필요하면 `.env.example`을 복사해 `.env`로 두고 `HF_TOKEN` 등을 넣습니다.

모델은 **먼저 저장소 안**(`models/mllm/`, `models/retriever/`)에서 찾고, 없으면 환경 변수나 저장소 기준 상대 경로(`models/`, `data/`)를 씁니다.

In [5]:
# data/model 설정 YAML이 저장소에 있는지 확인합니다.

from pathlib import Path

_cfgs = [REPO / "config/data.yaml", REPO / "config/model.yaml"]
_ok = 0
for p in _cfgs:
    ex = p.is_file()
    if ex:
        _ok += 1
    print(p.name, "있음:" if ex else "없음:", p)
print("설정 파일 준비:", f"{_ok}/{len(_cfgs)}")


data.yaml 있음: /workspace/late-interaction-mm-graph-rag/config/data.yaml
model.yaml 있음: /workspace/late-interaction-mm-graph-rag/config/model.yaml
설정 파일 준비: 2/2


## Hugging Face 모델 다운로드

가중치는 git 저장소에 **포함되어 있지 않습니다**. `util/download_models.py`로 `config/model.yaml`에 적힌 체크포인트를 `models/`로 받습니다(Gemma, ColEmbed, 메트릭용 선택 fluency BART).

- 모델이 게이트되어 있으면(Gemma) 저장소 루트 **`.env`**에 **`HF_TOKEN`** 또는 **`HUGGING_FACE_HUB_TOKEN`**을 넣습니다.
- **다운로드 후 구조:** `models/mllm/gemma-4-E4B-it/`, `models/retriever/llama-nemotron-colembed-vl-3b-v2/`, `models/eval/bart-large-cnn/`.
- **CLI:** `python util/download_models.py`(전체), 또는 `--only gemma colembed`, 또는 `--dry-run`(계획 경로만 출력).
- **Ollama**(예: `gemma4:e2b`)는 `models/`가 아니라 Ollama 기본 디렉터리에 받습니다. 아래 **Ollama 모델 다운로드** 절과 `config/model.yaml`의 `ollama` 항목을 참고하세요.

모델 이름·경로·HF 설정은 루트 **`README.md`**를 참고하세요.

In [ ]:
# config/model.yaml에 적힌 Gemma·ColEmbed 가중치를 Hugging Face에서 받습니다(시간·디스크 소요).

import os
import subprocess
import sys
from pathlib import Path

# Requires REPO from "Resolve repo root" above; fallback for quick runs
try:
    _r = REPO
except NameError:
    _here = Path.cwd()
    _r = _here if (_here / "util" / "download_models.py").is_file() else _here.parent

dl = _r / "util" / "download_models.py"
if not dl.is_file():
    raise FileNotFoundError(f"Missing {dl}")

print("=== Hugging Face 체크포인트 다운로드 (실행) ===")
print("스크립트: util/download_models.py")
print("대상: gemma, colembed (`config/model.yaml` 경로로 저장)")
print("주의: 시간이 길 수 있고, Gemma 등 게이트 모델은 `.env`의 HF_TOKEN 이 필요할 수 있습니다.")
print("아래에 huggingface_hub / 스크립트 로그가 그대로 출력됩니다.\n")

_dl_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_rc_dl = subprocess.run(
    [sys.executable, "-u", str(dl), "--only", "gemma", "colembed"],
    cwd=str(_r),
    env=_dl_env,
    check=False,
)
print("\n다운로드 프로세스 종료 코드:", _rc_dl.returncode)


## Ollama 모델 다운로드 (선택)

`config/model.yaml`의 **`ollama.model`**(예: `gemma4:e2b`)을 Ollama에 받습니다. 가중치는 **`models/`가 아니라** Ollama 기본 저장소(보통 **`~/.ollama`**)에 둡니다. 다른 루트는 환경 변수 **`OLLAMA_MODELS`**로 지정합니다.

`ollama pull`은 실행 중인 **Ollama 서버**가 있어야 합니다. **`could not connect to ollama server`**가 뜨면 **Ollama 서버** 단락부터 순서대로 실행하세요.

### Ollama 관련 셀 역할 (요약)

이 단락은 **저장소 기본 파이프라인(`pattern.py` / `extraction.py` / `inference.py`)이 쓰는 백엔드가 아닙니다.** 기본 설정은 **`VIDORE_TEXT_LLM_BACKEND=hf_gemma_4_e4b_it`**(Hugging Face Gemma in-process)이고, **Ollama**는 `config/model.yaml`의 `ollama` 태그를 쓰는 **선택 경로**(예: `mllm/ollama_*`, 데모 BE)용입니다. 여기서는 로컬 **Ollama 서버**에 모델을 받고, **REST**로 연결·스모크 Q&A만 확인합니다.

| 순서 | 내용 | 역할 |
|------|------|------|
| 1 | **설치·PATH 보강** | `ollama`가 `PATH`에 없을 때 흔한 `bin`을 앞에 붙이고, Linux면 공식 `install.sh`로 설치 시도(`zstd` 자동 설치 시도 포함). |
| 2 | `ollama serve` 셀 | API 주소(`OLLAMA_CONNECT_HOST`:`OLLAMA_PORT`, 기본 `127.0.0.1:11434`)가 열려 있는지 확인하고, 없으면 `ollama serve`를 백그라운드 기동 시도. |
| 3 | `util/download_models.py --only ollama` | `config/model.yaml`의 **`ollama.model`**, **`ollama.model_e4b`** 등에 맞춰 `ollama pull`. 가중치는 **`models/`가 아니라** Ollama 기본 저장소(보통 `~/.ollama`, `OLLAMA_MODELS`로 변경 가능). |
| 4 | `model.yaml` + `ollama list` | 설정 태그와 로컬 설치 여부 **진단**(pull 아님). |
| 5 | `~/.ollama` 용량 | `models`/`blobs` 디스크 사용량 참고용. |
| 6 | **`/api/chat` QA 셀** | **E2B·E4B 태그 각각** 짧은 질문 1회씩 호출하고, 응답마다 **wall time(초)** 출력. 파이프라인 추론과는 별개 스모크입니다. |


### Ollama 서버 (`ollama serve`)

서버 주소는 기본 **`127.0.0.1:11434`** 입니다. **`PATH`에 `ollama` CLI가 없으면** 바로 아래 코드 셀(**설치·PATH 보강**)을 먼저 실행한 뒤, **그 다음 셀**에서 백그라운드 `ollama serve`를 띄우세요. 터미널에서 이미 `ollama serve`를 켜 두었다면 설치 셀은 생략해도 됩니다(서버 셀이 이미 응답 중이면 건너뜀).


In [9]:
# 노트북 환경에서 ollama CLI를 찾게 PATH를 보강하고, Linux에서는 필요 시 설치 스크립트를 실행합니다.
import os
import platform
import shutil
import subprocess

# 이어지는 `ollama serve` 단계에서 `shutil.which("ollama")`가 통하도록 PATH를 앞쪽에 붙입니다.
_HOME = os.path.expanduser("~")
for _dir in ("/usr/local/bin", "/usr/bin", os.path.join(_HOME, ".local", "bin")):
    if _dir and os.path.isdir(_dir):
        _path = os.environ.get("PATH", "")
        if _dir not in _path.split(os.pathsep):
            os.environ["PATH"] = _dir + os.pathsep + _path if _path else _dir

# 설치만 되어 있고 노트북 세션 PATH에 없는 경우
for _cand in (
    "/usr/local/bin/ollama",
    "/usr/bin/ollama",
    os.path.join(_HOME, ".local", "bin", "ollama"),
):
    if os.path.isfile(_cand) and os.access(_cand, os.X_OK):
        _d = os.path.dirname(_cand)
        _path = os.environ.get("PATH", "")
        if _d not in _path.split(os.pathsep):
            os.environ["PATH"] = _d + os.pathsep + _path if _path else _d
        break

if shutil.which("ollama"):
    print("ollama CLI 확인:", shutil.which("ollama"))
else:
    _sys = platform.system()
    if _sys == "Linux":
        print("PATH에 ollama가 없어 공식 Linux 설치 스크립트를 실행합니다.")
        print("(네트워크 필요) https://ollama.com/download/linux")
        if not shutil.which("zstd"):
            print("Ollama 설치에 zstd가 필요합니다. apt/dnf/pacman으로 설치를 시도합니다…")
            _z = subprocess.run(
                [
                    "bash",
                    "-c",
                    "command -v zstd >/dev/null 2>&1 && exit 0; "
                    "if command -v apt-get >/dev/null 2>&1; then apt-get update -qq && apt-get install -y -qq zstd; "
                    "elif command -v dnf >/dev/null 2>&1; then dnf install -y zstd; "
                    "elif command -v pacman >/dev/null 2>&1; then pacman -S --noconfirm zstd; "
                    "else exit 1; fi",
                ],
            )
            if _z.returncode != 0 or not shutil.which("zstd"):
                raise RuntimeError(
                    "zstd가 없어 Ollama 설치 스크립트를 실행할 수 없습니다. 터미널에서 설치 후 이 셀을 다시 실행하세요.\n"
                    "  Debian/Ubuntu: sudo apt-get install zstd\n"
                    "  Fedora/RHEL:    sudo dnf install zstd\n"
                    "  Arch:           sudo pacman -S zstd"
                )
        _rc = subprocess.run(
            ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        )
        if _rc.returncode != 0:
            raise RuntimeError(
                "Ollama 자동 설치가 실패했습니다. zstd·curl 설치와 네트워크를 확인한 뒤 터미널에서:\n"
                "  curl -fsSL https://ollama.com/install.sh | sh\n"
                "완료 후 Jupyter 커널을 재시작하거나 `which ollama` 경로를 확인하세요."
            )
        for _dir in ("/usr/local/bin", "/usr/bin", os.path.join(_HOME, ".local", "bin")):
            if _dir and os.path.isdir(_dir):
                _path = os.environ.get("PATH", "")
                if _dir not in _path.split(os.pathsep):
                    os.environ["PATH"] = _dir + os.pathsep + _path if _path else _dir
        _rv = subprocess.run(
            ["bash", "-lc", "command -v ollama"],
            capture_output=True,
            text=True,
        )
        if _rv.returncode == 0:
            _op = _rv.stdout.strip()
            if _op:
                _od = os.path.dirname(_op)
                _path = os.environ.get("PATH", "")
                if _od not in _path.split(os.pathsep):
                    os.environ["PATH"] = _od + os.pathsep + _path if _path else _od
        if not shutil.which("ollama"):
            raise RuntimeError(
                "설치는 끝났지만 이 세션에서 ollama를 찾지 못했습니다. "
                "커널 재시작 후 이 셀과 다음 셀을 다시 실행하거나, 터미널에서 `which ollama`로 경로를 확인하세요."
            )
        print("설치 후 ollama CLI:", shutil.which("ollama"))
    elif _sys == "Darwin":
        raise RuntimeError(
            "macOS: https://ollama.com/download 에서 설치하거나 `brew install ollama` 후 "
            "커널을 재시작하고 이 셀을 다시 실행하세요."
        )
    else:
        raise RuntimeError(
            "https://ollama.com 에서 OS에 맞게 설치한 뒤 커널을 재시작하고 이 셀을 다시 실행하세요."
        )


ollama CLI 확인: /usr/local/bin/ollama


In [10]:
# Ollama HTTP API(기본 127.0.0.1:11434)가 살아 있는지 확인하고, 꺼져 있으면 `ollama serve`를 백그라운드로 띄웁니다.

import os
import shutil
import socket
import subprocess
import time

_CONNECT_HOST = os.getenv("OLLAMA_CONNECT_HOST", "127.0.0.1").strip() or "127.0.0.1"
_CONNECT_PORT = int((os.getenv("OLLAMA_PORT", "11434") or "11434").strip())


def _ollama_tcp_up(host: str, port: int, timeout: float = 2.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


if _ollama_tcp_up(_CONNECT_HOST, _CONNECT_PORT):
    print(f"Ollama 서버 이미 응답 중: {_CONNECT_HOST}:{_CONNECT_PORT}")
else:
    exe = shutil.which("ollama")
    if not exe:
        raise RuntimeError("PATH에 ollama CLI가 없습니다. https://ollama.com 에서 설치하세요.")

    prev = globals().get("OLLAMA_SERVER_PROC")
    if isinstance(prev, subprocess.Popen) and prev.poll() is None:
        print("OLLAMA_SERVER_PROC가 이미 떠 있음(PID {}). 연결 재시도…".format(prev.pid))
    else:
        print(f"백그라운드 시작: `{exe} serve` (접속 확인: {_CONNECT_HOST}:{_CONNECT_PORT}) …")
        out = subprocess.DEVNULL
        globals()["OLLAMA_SERVER_PROC"] = subprocess.Popen(
            [exe, "serve"],
            stdout=out,
            stderr=out,
            start_new_session=True,
        )
        print("PID:", globals()["OLLAMA_SERVER_PROC"].pid)

    for i in range(45):
        time.sleep(1)
        if _ollama_tcp_up(_CONNECT_HOST, _CONNECT_PORT):
            print(f"연결 OK — {_CONNECT_HOST}:{_CONNECT_PORT} (약 {i + 1}s)")
            break
    else:
        raise RuntimeError(
            f"{_CONNECT_HOST}:{_CONNECT_PORT} 에 연결할 수 없습니다. "
            "터미널에서 `ollama serve` 로그를 확인하거나 OLLAMA_PORT/방화벽을 점검하세요."
        )

Ollama 서버 이미 응답 중: 127.0.0.1:11434


In [7]:
# util/download_models.py --only ollama 로 model.yaml에 적힌 태그를 pull합니다(~/.ollama 등, repo models/ 아님).

import os
import subprocess
import sys
from pathlib import Path

try:
    _r = REPO
except NameError:
    _here = Path.cwd()
    _r = _here if (_here / "util" / "download_models.py").is_file() else _here.parent

dl = _r / "util" / "download_models.py"
if not dl.is_file():
    raise FileNotFoundError(f"Missing {dl}")

print("=== Ollama pull (실행) ===")
print("명령: python util/download_models.py --only ollama")
print("저장 위치: Ollama 기본 디렉터리(보통 ~/.ollama). repo models/ 아님.\n")

_dl_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_rc = subprocess.run(
    [sys.executable, "-u", str(dl), "--only", "ollama"],
    cwd=str(_r),
    env=_dl_env,
    check=False,
)
print("\n종료 코드:", _rc.returncode)

=== Ollama pull (실행) ===
명령: python util/download_models.py --only ollama
저장 위치: Ollama 기본 디렉터리(보통 ~/.ollama). repo models/ 아님.

--- ollama (default ~/.ollama store): gemma4:e2b
(Using Ollama default model directory; not writing under repo models/.)


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 4e30e2665218: 100% ▕██████████████████▏ 7.2 GB                         
pulling 7339fa418c9a: 100% ▕██████████████████▏  11 KB                         
pulling 56380ca2ab89: 100% ▕██████████████████▏   42 B                         
pulling c6bc3775a3fa: 100% ▕██████████████████▏  473 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ 

Done: ollama pull gemma4:e2b
--- ollama (default ~/.ollama store): gemma4:e4b
(Using Ollama default model directory; not writing under repo models/.)


pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ 

Done: ollama pull gemma4:e4b

Example environment overrides (local snapshots under models/):
  GEMMA4_E4B_IT_MODEL_PATH=/workspace/late-interaction-mm-graph-rag/models/mllm/gemma-4-E4B-it
  COLEMBED_MODEL_PATH=/workspace/late-interaction-mm-graph-rag/models/retriever/llama-nemotron-colembed-vl-3b-v2
  WEBQA_FLUENCY_MODEL=/workspace/late-interaction-mm-graph-rag/models/eval/bart-large-cnn

Ollama model 'gemma4:e2b': stored under the default Ollama directory (/root/.ollama), not repo models/.
Ollama model 'gemma4:e4b': stored under the default Ollama directory (/root/.ollama), not repo models/.

종료 코드: 0


pulling manifest ⠼ pulling manifest 
pulling 4c27e0f5b5ad: 100% ▕██████████████████▏ 9.6 GB                         
pulling 7339fa418c9a: 100% ▕██████████████████▏  11 KB                         
pulling 56380ca2ab89: 100% ▕██████████████████▏   42 B                         
pulling f0988ff50a24: 100% ▕██████████████████▏  473 B                         
verifying sha256 digest 
writing manifest 
success 


In [11]:
# model.yaml의 ollama.model 설정과 `ollama list`/`show` 출력을 맞춰 보는 점검용입니다(pull 실행 아님).

import os
import shutil
import subprocess
from pathlib import Path

try:
    import yaml
except ImportError:
    raise SystemExit("PyYAML 필요: pip install PyYAML")

try:
    _r = REPO
except NameError:
    _here = Path.cwd()
    _r = _here if (_here / "config" / "model.yaml").is_file() else _here.parent

cfg_path = _r / "config" / "model.yaml"
if not cfg_path.is_file():
    raise FileNotFoundError(cfg_path)

with cfg_path.open(encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}
oll_cfg = cfg.get("ollama") or {}
oll_m = (oll_cfg.get("model") or "").strip()

env_root = os.environ.get("OLLAMA_MODELS", "").strip()
default_root = Path.home() / ".ollama"
data_root = Path(env_root) if env_root else default_root

print("config/model.yaml → ollama.model:", oll_m or "(비어 있음)")
print("환경 변수 OLLAMA_MODELS:", env_root or "(미설정 → 기본 ~/.ollama)")
print("Ollama 데이터 루트(추정):", data_root.resolve())
print("  …/models 존재:", (data_root / "models").is_dir())

if shutil.which("ollama"):
    print("\n--- ollama list ---")
    subprocess.run(["ollama", "list"], check=False)
    if oll_m:
        print(f"\n--- ollama show {oll_m} ---")
        subprocess.run(["ollama", "show", oll_m], check=False)
else:
    print("\nPATH에 ollama CLI 없음.")

config/model.yaml → ollama.model: gemma4:e2b
환경 변수 OLLAMA_MODELS: (미설정 → 기본 ~/.ollama)
Ollama 데이터 루트(추정): /root/.ollama
  …/models 존재: True

--- ollama list ---
NAME          ID              SIZE      MODIFIED       
gemma4:e2b    7fbdbf8f5e45    7.2 GB    12 minutes ago    
gemma4:e4b    c6eb396dbd59    9.6 GB    12 minutes ago    

--- ollama show gemma4:e2b ---
  Model
    architecture        gemma4    
    parameters          5.1B      
    context length      131072    
    embedding length    1536      
    quantization        Q4_K_M    
    requires            0.20.0    

  Capabilities
    completion    
    vision        
    audio         
    tools         
    thinking      

  Parameters
    temperature    1       
    top_k          64      
    top_p          0.95    

  License
    Apache License               
    Version 2.0, January 2004    
    ...                          



In [12]:
# Ollama가 쓰는 데이터 루트(기본 ~/.ollama)에서 models/blobs 용량을 합산해 표시합니다.

from pathlib import Path
import os


def _ollama_data_root() -> Path:
    env = os.environ.get("OLLAMA_MODELS", "").strip()
    return Path(env) if env else Path.home() / ".ollama"


def _dir_size(p: Path) -> int:
    if not p.is_dir():
        return 0
    total = 0
    for f in p.rglob("*"):
        if f.is_file():
            try:
                total += f.stat().st_size
            except OSError:
                pass
    return total


def _human(n: int) -> str:
    x = float(max(n, 0))
    units = ("B", "KiB", "MiB", "GiB", "TiB")
    i = 0
    while x >= 1024 and i < len(units) - 1:
        x /= 1024
        i += 1
    return f"{int(x)} {units[i]}" if i == 0 else f"{x:.2f} {units[i]}"


root = _ollama_data_root()
models_dir = root / "models"
blobs = models_dir / "blobs"

print("Ollama 데이터 루트 (OLLAMA_MODELS 미설정 시 ~/.ollama):", root.resolve())
if models_dir.is_dir():
    sz = _dir_size(models_dir)
    print(f"  …/models 합계: {_human(sz)}  ({sz:,} bytes)")
else:
    print("  …/models 디렉터리 없음 (pull 전이거나 다른 루트)")

if blobs.is_dir():
    sz_b = _dir_size(blobs)
    nfiles = sum(1 for fp in blobs.rglob("*") if fp.is_file())
    print(f"  …/models/blobs 합계: {_human(sz_b)}  (파일 수: {nfiles})")
else:
    print("  …/models/blobs 없음")

Ollama 데이터 루트 (OLLAMA_MODELS 미설정 시 ~/.ollama): /root/.ollama
  …/models 합계: 15.62 GiB  (16,770,746,625 bytes)
  …/models/blobs 합계: 15.62 GiB  (파일 수: 6)


#### Ollama REST 스모크 QA (`/api/chat`)

아래 코드 셀은 **`config/model.yaml`의 `ollama.model`(E2B)과 `ollama.model_e4b`(E4B)** 에 대해 **같은 질문을 한 번씩** 보내고, 각 응답의 **소요 시간(wall time)** 을 출력합니다. 서버·`pull` 셀을 먼저 통과한 뒤 실행하세요.


In [ ]:
# Ollama `/api/chat`로 E2B·E4B 모델 태그마다 같은 질문을 한 번씩 보내고 응답까지 걸린 시간을 측정합니다.
# 메인 파이프라인 LLM은 HF Gemma이며, 여기서는 Ollama 연결 스모크만 합니다.
import json
import os
import time
from pathlib import Path

import requests
import yaml

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "config" / "model.yaml").is_file() else _here.parent

with (_repo / "config" / "model.yaml").open(encoding="utf-8") as f:
    _model_cfg = yaml.safe_load(f) or {}
_ollama = _model_cfg.get("ollama") or {}
_model_e2b = str(_ollama.get("model") or "gemma4:e2b").strip()
_model_e4b = str(_ollama.get("model_e4b") or "gemma4:e4b").strip()

_host = os.getenv("OLLAMA_CONNECT_HOST", "127.0.0.1").strip() or "127.0.0.1"
_port = int((os.getenv("OLLAMA_PORT", "11434") or "11434").strip())
_url = f"http://{_host}:{_port}/api/chat"

_question = "프랑스의 수도는 어디인가요? 한 문장으로 답하세요."

_done_tags: set[str] = set()
for _title, _model in (
    ("ollama.model (E2B)", _model_e2b),
    ("ollama.model_e4b (E4B)", _model_e4b),
):
    if not _model or _model in _done_tags:
        continue
    _done_tags.add(_model)
    print(f"\n--- {_title}: {_model} ---")
    _body = {
        "model": _model,
        "messages": [{"role": "user", "content": _question}],
        "stream": False,
    }
    _t0 = time.perf_counter()
    try:
        r = requests.post(_url, json=_body, timeout=180)
    except requests.exceptions.RequestException as e:
        raise RuntimeError(
            f"Ollama API 연결 실패 ({_url}). 위에서 Ollama 서버·pull을 먼저 실행했는지 확인하세요."
        ) from e
    r.raise_for_status()
    _out = r.json()
    _reply = (_out.get("message") or {}).get("content", "").strip()
    _elapsed_s = time.perf_counter() - _t0
    print("질문:", _question)
    print("응답:", _reply or json.dumps(_out, ensure_ascii=False)[:500])
    print(f"소요 시간: {_elapsed_s:.2f}s ({_elapsed_s * 1000:.0f} ms)")

## 간단한 텍스트 QA

**Gemma 4 E4B IT**(위 Hugging Face 스냅샷)을 올려 질문에 답을 생성합니다.

- 첫 실행은 가중치 로딩에 시간이 걸릴 수 있습니다. GPU가 있으면 셀에서 `bf16`을 기본으로 잡습니다.
- 가중치가 없으면 위 **Hugging Face 모델 다운로드** 셀을 먼저 실행하세요.


In [ ]:
# Hugging Face Gemma(in-process)로 짧은 질문에 답을 생성해 봅니다(첫 실행은 가중치 로딩에 시간이 걸릴 수 있음).
# 사전 조건: "저장소 루트와 경로 맞추기" 셀을 실행해 `REPO`가 있어야 합니다.
import logging
import os
import sys
import time
from pathlib import Path

if "REPO" not in globals():
    raise RuntimeError("먼저 REPO를 정의하는 셀을 실행하세요.")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# 모델 로딩·HF 스택에서 나오는 로그를 노트북 출력으로 보이게 함
def _setup_verbose_logging() -> None:
    fmt = logging.Formatter("%(levelname)s [%(name)s] %(message)s")
    root = logging.getLogger()
    if not any(isinstance(h, logging.StreamHandler) and getattr(h, "stream", None) is sys.stdout for h in root.handlers):
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(fmt)
        root.addHandler(h)
    root.setLevel(logging.INFO)
    for name in ("mllm.hf_gemma_4_e4b_it", "transformers", "transformers.modeling_utils"):
        logging.getLogger(name).setLevel(logging.INFO)
    try:
        from transformers import logging as tf_logging

        tf_logging.set_verbosity_info()
    except Exception:
        pass


_setup_verbose_logging()

import torch
from util.llm_defaults import effective_gemma4_e4b_it_model_path
from mllm.hf_gemma_4_e4b_it import configured, generate_text

_gemma_dir = effective_gemma4_e4b_it_model_path()
print("Gemma 가중치 경로:", _gemma_dir)
print("torch / CUDA:", torch.__version__, "|", torch.cuda.is_available())
print("가중치 사용 가능(configured):", configured())

if not configured():
    print("가중치 디렉터리가 없습니다. 위 'Hugging Face 모델 다운로드' 셀을 실행한 뒤 다시 시도하세요.")
else:
    if torch.cuda.is_available():
        os.environ.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")
        print("dtype 힌트: GEMMA4_E4B_IT_TORCH_DTYPE =", os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE", "(default)"))

    question = "What is the capital of France? Answer in one short sentence."
    print("\n--- 질문 ---\n", question)
    print("\n--- 응답 생성 중: 첫 실행 시 아래에 모델·프로세서 로딩 로그가 찍힌 뒤 generate_text가 진행됩니다 ---\n")
    t0 = time.perf_counter()
    answer = generate_text(question, max_new_tokens=128)
    elapsed = time.perf_counter() - t0
    print("\n--- 답변 ---\n", answer)
    print(f"\n--- 끝 (wall time {elapsed:.2f}s) ---")


## 토이 데이터셋(shard 14)

`data/webqa/webqa_shard14_toy/webqa_slice/`가 이미 저장소에 있으면 **다시 만들지 않아도** 됩니다.

없으면 토이 슬라이스를 생성합니다(전체 WebQA + `scripts/build_shard14_toy_split.py` 기본값과 같은 베이스라인 파일 필요):

```bash
python scripts/build_shard14_toy_split.py
```

PNG는 `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/` 등에 있거나 `webqa_images.jsonl`에 기록된 경로를 가리킵니다. 아래 **토이 이미지 미리보기**를 실행하면 노트북에서 몇 장을 볼 수 있습니다.

In [ ]:
# 토이 webqa_slice의 JSONL 줄 수·Qcate 분포 등을 집계해 데이터 형태를 점검합니다.

TOY_SLICE = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

print("=== 토이 데이터셋(WebQA 슬라이스) ===")
print("데이터셋: WebQA — shard 14 **toy** 슬라이스 (미니 코퍼스, 질의·이미지·텍스트 JSONL)")
print("경로:", TOY_SLICE.resolve())
print("일반 구성: webqa_questions.jsonl(질문·정답·메타), webqa_texts.jsonl(텍스트 청크), webqa_images.jsonl(이미지 인덱스), webqa_tables.jsonl(선택)")

_names = (
    "webqa_questions.jsonl",
    "webqa_texts.jsonl",
    "webqa_images.jsonl",
    "webqa_tables.jsonl",
)
for _name in _names:
    _p = TOY_SLICE / _name
    if _p.is_file():
        _n = sum(1 for _ in _p.open(encoding="utf-8"))
        print(f"  {_name}: {_n}줄")
    else:
        print(f"  {_name}: (파일 없음)")

_qp = TOY_SLICE / "webqa_questions.jsonl"
if _qp.is_file():
    import json as _json
    from collections import Counter

    _qcates = Counter()
    _splits = Counter()
    _n_img_only = _n_txt_only = _n_both = _n_neither = 0
    _n_chars_q = 0
    _rows = 0
    for _line in _qp.open(encoding="utf-8"):
        _line = _line.strip()
        if not _line:
            continue
        _rows += 1
        _o = _json.loads(_line)
        _qcates[str(_o.get("Qcate") or "?")] += 1
        _splits[str(_o.get("split") or "?")] += 1
        _q = _o.get("Q") or ""
        _n_chars_q += len(_q)
        _hi = bool(_o.get("img_posFacts"))
        _ht = bool(_o.get("txt_posFacts"))
        if _hi and _ht:
            _n_both += 1
        elif _hi:
            _n_img_only += 1
        elif _ht:
            _n_txt_only += 1
        else:
            _n_neither += 1
    print("질문 JSONL 상세 (webqa_questions.jsonl):")
    print("  레코드 수:", _rows)
    if _rows:
        print("  질문 길이 평균(글자 수):", round(_n_chars_q / _rows, 1))
    print("  Qcate별 개수:", dict(sorted(_qcates.items())))
    print("  split별 개수:", dict(sorted(_splits.items())))
    print(
        "  긍정 증거 타입 — 이미지+텍스트:",
        _n_both,
        "| 이미지만:",
        _n_img_only,
        "| 텍스트만:",
        _n_txt_only,
        "| 둘 다 없음:",
        _n_neither,
    )
else:
    print("webqa_questions.jsonl 없음 — 위 상세 통계를 계산할 수 없습니다.")


#### WebQA 예시 — 질문·정답 (gold)

`data/webqa/.../webqa_slice/webqa_questions.jsonl` 한 줄이 **한 개의 질문 레코드**입니다(WebQA 멀티모달 QA 코퍼스에서 쓰는 JSONL 계열과 동일한 필드 관례).

| 필드 | 역할 |
|------|------|
| **`Guid`** | 질문 id (`qid` 역할). |
| **`Q`** | 질문 본문. |
| **`A`** | **Gold 정답** 문자열 리스트(보통 1개). |
| **`Qcate`** | 질문 유형 — 평가(`eval/evaluate_webqa_qa.py`)에서 **키워드·정확도 규칙**에 사용됩니다(예: `YesNo`, `choose`, …). |
| **`Keywords_A`** | QA-Acc 등에서 쓰는 키워드 힌트. |
| **`metadata.image_doc_ids` / `text_doc_ids`** | 멀티모달 문맥 id — 이후 **pattern / extraction / construct**가 연결하는 단위입니다. |

아래 코드는 토이 슬라이스에서 **서로 다른 `Qcate`를 우선**으로 골라 **3건 이상**을 출력합니다.


In [ ]:
# 토이 webqa_questions.jsonl에서 Qcate를 고르듯이 골라 질문·gold(A)·메타를 여러 건 출력합니다.

import json
from pathlib import Path

# 사전 조건: 바로 위 셀에서 TOY_SLICE 가 정의되어 있어야 합니다.
_qp = TOY_SLICE / "webqa_questions.jsonl"
if not _qp.is_file():
    print("webqa_questions.jsonl 없음 — gold 예시를 건너뜁니다:", _qp)
else:
    _rows: list[dict] = []
    for _line in _qp.open(encoding="utf-8"):
        _line = _line.strip()
        if _line:
            _rows.append(json.loads(_line))

    def _print_webqa_gold(o: dict, title: str) -> None:
        _guid = o.get("Guid", "")
        _q = o.get("Q") or ""
        _a = o.get("A") or []
        _qcate = o.get("Qcate", "")
        _kw = o.get("Keywords_A", "")
        _meta = o.get("metadata") or {}
        _img = _meta.get("image_doc_ids") or []
        _txt = _meta.get("text_doc_ids") or []
        print(f"\n=== {title} ===")
        print("Guid:", _guid)
        print("Qcate:", _qcate)
        print("질문(Q):", _q[:500] + ("…" if len(_q) > 500 else ""))
        print("정답(gold) A:", _a)
        print("Keywords_A:", _kw)
        _ts = _txt[:5]
        print(
            "metadata — image_doc_ids:",
            _img,
            "| text_doc_ids (앞 5개):",
            _ts,
            ("…" if len(_txt) > 5 else ""),
        )

    # 1) Qcate 당 1건씩(최대 5유형) → 다양한 gold 유형 노출
    _by_q: dict[str, dict] = {}
    for _o in _rows:
        _c = str(_o.get("Qcate") or "?")
        if _c not in _by_q:
            _by_q[_c] = _o
    _picked: list[dict] = list(_by_q.values())

    # 2) 3건 미만이면 파일 순서로 채움
    _i = 0
    while len(_picked) < 3 and _i < len(_rows):
        if _rows[_i] not in _picked:
            _picked.append(_rows[_i])
        _i += 1

    # 3) 3건 이상 보장 후, 총 **5건**까지(또는 전체가 더 적으면 모두)
    for _o in _rows:
        if len(_picked) >= 5:
            break
        if _o not in _picked:
            _picked.append(_o)

    print(f"표시할 gold 예시: {len(_picked)}건 (전체 레코드 {len(_rows)}건, Qcate 종류 {len(_by_q)}가지)")
    for _j, _o in enumerate(_picked, start=1):
        _print_webqa_gold(_o, f"WebQA gold 예시 {_j}")


### 토이 이미지 미리보기(선택)

`webqa_images.jsonl`을 읽어 PNG 몇 장을 표시합니다. JSONL 안 경로가 다른 환경의 절대 경로를 가리키더라도 파일이 없으면, 같은 파일명으로 저장소 안 샤드 폴더 `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/`를 시도합니다.

In [ ]:
# webqa_images.jsonl을 몇 줄 읽어 실제 PNG 경로를 풀어 노트북에 표시합니다.

import json
from pathlib import Path

from IPython.display import Image as IPyImage, display

try:
    _slice = TOY_SLICE
except NameError:
    try:
        _slice = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"
    except NameError:
        _here = Path.cwd()
        _root = _here if (_here / "data" / "webqa").is_dir() else _here.parent
        _slice = _root / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "inference.py").is_file() else _here.parent

imgs_jsonl = _slice / "webqa_images.jsonl"
local_shard = _repo / "data" / "webqa" / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014"

if not imgs_jsonl.is_file():
    print("없음:", imgs_jsonl)
else:
    rows = []
    with imgs_jsonl.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print(f"슬라이스 인덱스 이미지 수: {len(rows)}")
    for obj in rows[:5]:
        url = obj.get("url") or ""
        p = Path(url)
        if not p.is_file() and p.name:
            alt = local_shard / p.name
            if alt.is_file():
                p = alt
        if not p.is_file():
            print("건너뜀(파일 없음):", url[:80], "...")
            continue
        print(obj.get("image_id"), obj.get("title", "")[:60])
        display(IPyImage(filename=str(p), width=400))

## 처음부터 끝까지 한 번에 실행(학습자 권장)

`tests/test_pipeline.py`가 올바른 환경 변수로 0→6 단계를 묶어 실행합니다. 터미널에서 돌릴 때와 같습니다.

**단계별 수동 실행**을 쓰는 경우 이 절은 **건너뛰어도** 됩니다.

- **`-n 5`** — 질문 5개(빠름).
- **`--dry-run`** — GPU LLM 없음; 자리 표시 답(배선 확인용).
- **`GEMMA4_E4B_IT_TORCH_DTYPE=bf16`** — VRAM이 부족할 때 실제 실행에 권장.

In [ ]:
# tests/test_pipeline.py로 전체 파이프라인을 한 번에 실행합니다(DRY_RUN·N_QUERIES 조절 가능).

import os
import subprocess
import sys

# 기본은 실추론(GPU). True면 드라이런(스모크·CPU 친화)
DRY_RUN = False  # 기본: GPU·실모델 (빠른 스모크만: True)
N_QUERIES = 2

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)
if not DRY_RUN:
    env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

cmd = [
    sys.executable,
    str(REPO / "tests" / "test_pipeline.py"),
    "-n", str(N_QUERIES),
]
if DRY_RUN:
    cmd.append("--dry-run")

print("통합 파이프라인 설정  DRY_RUN=", DRY_RUN, " N_QUERIES=", N_QUERIES)
print("실행 명령:", " ".join(cmd))
r = subprocess.run(cmd, cwd=str(REPO), env=env)
print("종료 코드:", r.returncode)

# 통합 테스트 후 최근 result/ 폴더와 산출물 존재 여부 안내(webqa·multimodalqa 하위 RUN_ID)
print("\n--- 통합 실행 결과 위치(참고) ---")


def _collect_run_dirs(result_root):
    out = []
    for bucket in ("webqa", "multimodalqa"):
        b = result_root / bucket
        if b.is_dir():
            for p in b.iterdir():
                if p.is_dir():
                    out.append(p)
    for p in result_root.iterdir():
        if p.is_dir() and p.name not in ("webqa", "multimodalqa") and (p / "phase5_inference").is_dir():
            out.append(p)
    return out


_rr = REPO / "result"
if _rr.is_dir():
    _runs = sorted(_collect_run_dirs(_rr), key=lambda p: p.stat().st_mtime, reverse=True)
    for _p in _runs[:5]:
        _pred = _p / "phase5_inference" / "predictions.json"
        _rep = _p / "phase5_inference" / "evaluation_report.json"
        _flags = []
        if _pred.is_file():
            _flags.append("predictions")
        if _rep.is_file():
            _flags.append("evaluation_report")
        _tag = ", ".join(_flags) if _flags else "(추론/평가 산출물 없음)"
        _rel = _p.relative_to(_rr)
        print(f"  {_rel}  |  {_tag}")
    if _runs:
        _latest_rep = _runs[0] / "phase5_inference" / "evaluation_report.json"
        if _latest_rep.is_file():
            import json as _json

            def _sf(_x):
                try:
                    return float(_x) if _x is not None else 0.0
                except (TypeError, ValueError):
                    return 0.0

            with _latest_rep.open(encoding="utf-8") as _f:
                _repd = _json.load(_f)
            _all_lb = (_repd.get("leaderboard_summary") or {}).get("All") or {}
            if _all_lb:
                print("\n--- 가장 최근 result 폴더의 평가 점수(전체 All) ---")
                print(
                    f"  QA-FL={_sf(_all_lb.get('QA-FL')):.4f}  "
                    f"QA-Acc={_sf(_all_lb.get('QA-Acc')):.4f}  "
                    f"QA={_sf(_all_lb.get('QA')):.4f}"
                )
                print("  리포트:", _latest_rep.resolve())
else:
    print("  result/ 폴더가 없습니다. 파이프라인을 한 번 실행해 보세요.")



실행이 끝나면 결과는 **`result/<RUN_ID>/`** 아래에 생깁니다:

- **`webqa_slice/`** — 이번 run이 실제로 읽는 **실행용 번들**. 질문·문맥 JSONL(`webqa_questions.jsonl`, `webqa_texts.jsonl`, `webqa_images.jsonl`, 필요 시 `webqa_tables.jsonl`)이 표준 이름으로 들어 있습니다.
- **`phase2_pattern_cache/`** — 질문별(`Guid` 등) LLM **패턴** JSON. 3단계 추출이 따르는 설계도.
- **`phase3_extraction_cache/`** — 추출 결과 JSON.
- **`phase4_graphs_out/*.graphml`** — 구축된 지역 그래프.
- **`phase5_inference/predictions.json`**, **`evaluation_report.json`** — 모델 답변 및 Phase 6 평가 리포트(있을 때).

In [ ]:
# result/ 아래 최근 실행 폴더와 predictions·evaluation_report 존재 여부를 나열합니다.
# 산출물 경로는 result/webqa/<RUN_ID>/ 및 result/multimodalqa/<RUN_ID>/ 입니다(바로 아래 RUN_ID가 아님).

import datetime


def _collect_run_dirs(result_root):
    """Runs live under result/webqa/ and result/multimodalqa/; optionally legacy result/<RUN_ID>/."""
    out = []
    for bucket in ("webqa", "multimodalqa"):
        b = result_root / bucket
        if b.is_dir():
            for p in b.iterdir():
                if p.is_dir():
                    out.append(p)
    for p in result_root.iterdir():
        if p.is_dir() and p.name not in ("webqa", "multimodalqa") and (p / "phase5_inference").is_dir():
            out.append(p)
    return out


result_root = REPO / "result"
if result_root.is_dir():
    runs = sorted(_collect_run_dirs(result_root), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in runs[:5]:
        ts = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        pred = (p / "phase5_inference" / "predictions.json").is_file()
        rep = (p / "phase5_inference" / "evaluation_report.json").is_file()
        rel = p.relative_to(result_root)
        print(rel, "| 수정 시각:", ts.isoformat(sep=" ", timespec="seconds"))
        print("    predictions:", pred, "| evaluation_report:", rep)
    if not runs:
        print("(result/webqa 또는 result/multimodalqa 아래 실행 폴더가 없습니다)")
else:
    print("아직 result/ 없음 — 먼저 파이프라인을 한 번 실행하세요.")



## GraphML 하나 살펴보기(4단계 출력)

그래프는 일반 NetworkX GraphML이며, **nodes** · **edges** 용어는 아래 출력에서 영어 그대로 둡니다. 각 node에는 `entity_name`, `type`, `description`, `source_id` 등의 속성이 붙습니다.


**노드(node)**는 그래프의 **점**이며, 여기서는 질문·답 근거가 되는 **엔티티**(예: 질문 앵커 `QUESTION`, 텍스트 조각 `TEXT`, 이미지 `IMAGE` 등) 한 덩어리입니다. GraphML에는 `type`, `entity_name`, `description`, `source_id`가 붙습니다.

**엣지(edge)**는 **노드 사이의 링크**이며, 두 엔티티가 **어떤 관계로 연결됐는지**를 나타냅니다. 속성으로 `description`(관계 의미, 예: `related context`), `weight`, `source_id` 등이 올 수 있습니다.

**이 절의 코드 셀:** `nx.read_graphml`로 **result/webqa** 산출물 중 가장 최근 수정된 GraphML 한 개를 읽고, 통계를 출력한 뒤 **matplotlib**으로 spring layout **정적 미리보기**를 그립니다(노드가 너무 많으면 plot은 건너뜁니다).

**대화형 시각화:** 아래 **데모 UI(선택)** 절에서 FE를 띄우면, 브라우저에서 질문별 그래프를 **force-directed**로 탐색할 수 있습니다. 구현은 Vite 기반 FE의 `GraphViewer` 컴포넌트(`react-force-graph-2d`)가 백엔드에서 받은 nodes/edges를 그립니다.




In [ ]:
# result/webqa 아래 phase4 GraphML 중 하나를 골라 통계와 spring layout 미리보기를 만듭니다.

import matplotlib.pyplot as plt
import networkx as nx

webqa_root = REPO / "result" / "webqa"
gml_files = list(webqa_root.glob("**/phase4_graphs_out/*.graphml"))
if not gml_files:
    print("No GraphML under result/webqa/*/phase4_graphs_out/")
else:
    path = max(gml_files, key=lambda p: p.stat().st_mtime)
    G = nx.read_graphml(path)
    print("file:", path.relative_to(REPO))
    print("nodes:", G.number_of_nodes(), "  edges:", G.number_of_edges())
    try:
        _Gu = G.to_undirected() if G.is_directed() else G
        print("density (undirected):", f"{nx.density(_Gu):.6f}")
    except Exception as _e:
        print("density (undirected) skipped:", _e)

    H = _Gu

    def _short(text, lim=36):
        if text is None:
            return ""
        text = str(text).replace("\n", " ")
        return text if len(text) <= lim else text[: lim - 1] + "…"

    print("\n--- Node = 점(엔티티): 질문·텍스트·이미지 등 한 단위 ---")
    _ni = 0
    for n in H.nodes:
        if _ni >= 14:
            print(f"  ... 이후 노드 {H.number_of_nodes() - _ni}개 생략")
            break
        d = H.nodes[n]
        print(
            "  [{}] type={!r}  entity={}".format(
                _ni + 1,
                d.get("type"),
                _short(d.get("entity_name") or str(n), 72),
            )
        )
        if d.get("description"):
            print("       desc :", _short(d.get("description"), 78))
        _ni += 1

    print("\n--- Edge = 링크(관계): 두 엔티티 연결 + 의미 ---")
    _ej = 0
    for u, v, dd in H.edges(data=True):
        if _ej >= 16:
            print(f"  ... 이후 엣지 {H.number_of_edges() - _ej}개 생략")
            break
        print(
            "  [{}] {}  --[{}] w={}--  {}".format(
                _ej + 1,
                _short(str(u), 34),
                dd.get("description") or "",
                dd.get("weight"),
                _short(str(v), 34),
            )
        )
        _ej += 1

    # quick layout (large graphs: layout can be slow)
    _G = G.to_undirected() if G.is_directed() else G
    _n = _G.number_of_nodes()
    if _n == 0:
        pass
    elif _n > 300:
        print("plot skipped: too many nodes (", _n, ") — open Gephi/GraphML tools for big graphs")
    else:
        fig, ax = plt.subplots(figsize=(7, 5), dpi=110)
        k = 0.6 / max(1, _n ** 0.5)
        pos = nx.spring_layout(_G, seed=0, k=k, iterations=50)
        nx.draw_networkx(
            _G,
            pos=pos,
            ax=ax,
            with_labels=False,
            node_size=180,
            width=0.6,
            edge_color="gray",
            alpha=0.9,
        )
        ax.set_title("WebQA — GraphML preview (spring layout)")
        ax.axis("off")
        fig.tight_layout()
        plt.show()



## 단계별 실행(수동)

다음 셀에서 `RUN_ID`, `N_QUERIES`, `DRY_RUN`을 정한 뒤 0→6 단계를 순서대로 실행합니다. `tests/test_pipeline.py`와 같은 설정(토이 데이터, 로컬 경로)입니다.

**슬라이스(0단계):** 아래 셀을 실행하면 **`result/webqa/<YYYYMMDD_HHMMSS>_<RUN_SLUG>/webqa_slice/`**에 토이 JSONL을 두게 됩니다(`MMGRAPHRAG_RUN_ID`는 `webqa/<같은이름>/` 형식). 이후 pattern·extraction·construct·inference env는 그 경로를 가리킵니다. **`N_QUERIES`**는 slice 안의 전체 질문 수를 바꾸지 않고, **앞에서부터 몇 줄만** 파이프라인에 태울지를 제한합니다.

**같은 실행을 이어서** 할 때는 **준비 셀을 다시 실행하지 말고** 기존 `RUN_ID`/`result_dir`를 유지하세요. **새 폴더**로 돌리려면 준비 셀을 다시 실행하거나 `RUN_SLUG`만 바꿉니다.



In [ ]:
# 단계별 수동 실행에 쓸 RUN_ID·env(common_env)·run_step을 준비합니다.
# 토이 슬라이스 경로·이미지 샤드·Gemma/ColEmbed 경로를 한곳에 묶습니다.

import os
import shutil
import subprocess
import sys
from pathlib import Path

# --- 실험에서 바꿀 값 ---
# 결과: result/webqa/<YYYYMMDD_HHMMSS>_<RUN_SLUG>/ — util.result_layout, tests/test_pipeline.py와 동일
from util.result_layout import webqa_stamped_run_id

RUN_SLUG = "notebook_tutorial"  # 실험별 접미사만 바꾸면 됨
RUN_ID = webqa_stamped_run_id(RUN_SLUG)  # 이 셀을 실행할 때마다 새 날짜·시간 prefix
# slice(webqa_slice/) 전체 크기와 별개: pattern~inference에서 처리할 질문 수 상한
N_QUERIES = 2
DRY_RUN = False  # 기본: GPU·Gemma 실추론 (스모크/CPU만: True)

# --- 경로(tests/test_pipeline.py와 동일 로직) ---
_LOCAL_DATA = REPO / "data" / "webqa"
if (_LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice").is_dir():
    TOY_SLICE = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice"
else:
    TOY_SLICE = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

if (_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014").is_dir():
    SHARD14_IMGS = str(_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014")
else:
    SHARD14_IMGS = str(REPO / "data" / "webqa" / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014")

result_dir = REPO / "result" / RUN_ID
slice_dir = result_dir / "webqa_slice"
q_file = slice_dir / "webqa_questions.jsonl"
t_file = slice_dir / "webqa_texts.jsonl"

dry_run = "1" if DRY_RUN else "0"

from util.llm_defaults import DEFAULT_GEMMA4_E4B_IT_MODEL_PATH
gemma_path = os.environ.get("GEMMA4_E4B_IT_MODEL_PATH", "").strip() or str(DEFAULT_GEMMA4_E4B_IT_MODEL_PATH)

common_env = {
    "PYTHONPATH": str(REPO),
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "WEBQA_RUN_PROFILE": "val_n100",
    "WEBQA_DATA_ROOT": str(_LOCAL_DATA) if _LOCAL_DATA.is_dir() else str(REPO / "data" / "webqa"),
    "PATTERN_MAX_SAMPLES": str(N_QUERIES),
    "WEBQA_EXPORT_MAX": str(N_QUERIES),
    "EXTRACTION_MAX_QUESTIONS": str(N_QUERIES),
    "CONSTRUCT_MAX_QUESTIONS": str(N_QUERIES),
    "PATTERN_DRY_RUN": dry_run,
    "EXTRACTION_DRY_RUN": dry_run,
    "PATTERN_CONCURRENCY": "1",
    "EXTRACTION_CONCURRENCY": "1",
    "GEMMA4_E4B_IT_MODEL_PATH": gemma_path,
    "VIDORE_TEXT_LLM_BACKEND": "hf_gemma_4_e4b_it",
    "COLEMBED_MODEL_PATH": os.environ.get(
        "COLEMBED_MODEL_PATH",
        str(REPO / "models" / "retriever" / "llama-nemotron-colembed-vl-3b-v2"),
    ),
}
if os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE"):
    common_env["GEMMA4_E4B_IT_TORCH_DTYPE"] = os.environ["GEMMA4_E4B_IT_TORCH_DTYPE"]
elif not DRY_RUN:
    common_env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

def run_step(desc: str, cmd: list[str], env: dict) -> int:
    merged = {**os.environ, **env}
    print(f"\n=== {desc} ===\n{' '.join(cmd)}\n")
    r = subprocess.run(cmd, cwd=str(REPO), env=merged)
    print(f"종료 코드={r.returncode}")
    return r.returncode

print("RUN_ID:", RUN_ID, "| N_QUERIES:", N_QUERIES, "| DRY_RUN:", DRY_RUN)
print("토이 슬라이스:", TOY_SLICE)

print("결과 디렉터리:", result_dir)
print("슬라이스(다음 셀에서 복사됨):", slice_dir)
print("이미지 샤드:", SHARD14_IMGS)
print("Gemma 경로:", gemma_path)
print("ColEmbed:", common_env.get("COLEMBED_MODEL_PATH"))



### 0단계 — 토이 `webqa_slice`를 `result/<RUN_ID>/`로 복사

여기서도 **slice**는 원본(`data/webqa/.../webqa_slice`)과 동일한 내용을 **`result/<RUN_ID>/webqa_slice/`**에 두는 **실행용 번들**입니다. downstream 스크립트는 항상 run 폴더 아래 표준 이름만 보므로, 경로를 한곳으로 고정합니다.

더 큰 코퍼스를 쓸 때는 **`export_webqa_slice.py`**로 같은 식의 `webqa_slice/`를 만들 수 있고, 이 수동 절에서는 **이미 만들어진 토이 JSONL**을 `shutil.copytree`로 옮기는 형태입니다.

In [ ]:
# data/webqa/.../webqa_slice 내용을 result/<RUN_ID>/webqa_slice 로 그대로 복사해 0단계 슬라이스를 만듭니다.

if not TOY_SLICE.is_dir():
    raise FileNotFoundError(f"토이 슬라이스 없음: {TOY_SLICE}")

result_dir.mkdir(parents=True, exist_ok=True)
if slice_dir.exists():
    shutil.rmtree(slice_dir)
shutil.copytree(TOY_SLICE, slice_dir)
print("복사 완료:", slice_dir)
_qpath = slice_dir / "webqa_questions.jsonl"
if _qpath.is_file():
    _nq = sum(1 for _ in _qpath.open(encoding="utf-8"))
    print("슬라이스 질문 줄 수(webqa_questions.jsonl):", _nq)
else:
    print("(webqa_questions.jsonl 없음)")

### 패턴 (`pattern.py`)

**패턴**은 질문 문장의 의미에 맞춰, **지역 지식 그래프를 어떤 식으로 짤지**를 정하는 **부분 그래프 골격**입니다. 2단계 `pattern.py`는 WebQA 레코드의 **`Q`**(또는 MMQA의 `question`)를 프롬프트에 넣어 LLM으로부터 **엔티티 타입·관계 타입** 후보와 연결 형식을 받습니다.

- **Pattern cache**는 질문 하나당 하나의 JSON(`phase2_pattern_cache/{Guid}.json` 등)으로 저장됩니다. 입력은 **질문 텍스트**이고, 출력 문자열은 구조화된 **그래프 패턴**입니다.
- 3단계 **extraction**은 이 패턴을 **constraint**로 삼아 `webqa_texts.jsonl` / `webqa_images.jsonl` 등에서 엔티티·관계를 채웁니다.
- 이 cache는 데이터셋 **정답(`A`)이 아니며**, 그래프를 만들기 위한 **질의 주도 스키마**(query-driven schema)에 가깝습니다. 잘못된 패턴이면 이후 단계 품질이 망가질 수 있어, 아래 **패턴 캐시 미리보기** 셀로 형태를 확인하는 것이 좋습니다.

**실행 순서:** 아래 **2단계 전 점검·쿼리 상한** 코드 셀에서 `N_QUERIES`와 경로를 확정한 뒤, **2단계 `pattern.py`** 셀을 실행합니다.




In [ ]:
# pattern.py가 각 질문에 대해 사용하는 프롬프트 템플릿: `prompt.GRAPH_PATTERN_PROMPT`
# `{question}` 자리에 WebQA 레코드의 **`Q`** 필드 문자열이 치환됩니다 (`pattern.process_batch`).

from prompt import GRAPH_PATTERN_PROMPT

print("GRAPH_PATTERN_PROMPT — pattern.py의 LLM 입력 템플릿\n")
print(GRAPH_PATTERN_PROMPT)


In [ ]:
# 2단계 전 — 점검, webqa_slice 준비, 쿼리 개수(상한) 설정·출력
# 위 **준비** 셀을 실행한 뒤 이 셀을 실행하세요.
# 숫자는 여기에서 최종 확정됩니다(pattern·export·추출·구축 상한을 common_env에 반영).

import shutil
from pathlib import Path

N_QUERIES = 2  # ← 처리할 최대 질문 수 — 필요하면 숫자만 바꿉니다.

_req = ("REPO", "result_dir", "slice_dir", "common_env", "run_step", "_LOCAL_DATA", "TOY_SLICE")
_missing = [n for n in _req if n not in globals()]
if _missing:
    raise RuntimeError(
        "다음이 정의되지 않았습니다: "
        + ", ".join(_missing)
        + "\n→ **저장소 루트(REPO)** 셀과 **단계별 수동 실행 준비** 셀을 먼저 실행하세요."
    )

_slice_q = Path(slice_dir) / "webqa_questions.jsonl"
if not _slice_q.is_file():
    if TOY_SLICE.is_dir():
        print("(자동 설정) webqa_slice 없음 — 토이 슬라이스를 이번 run으로 복사합니다(0단계와 동일).")
        print("     ", TOY_SLICE, " → ", slice_dir)
        result_dir.mkdir(parents=True, exist_ok=True)
        if slice_dir.exists():
            shutil.rmtree(slice_dir)
        shutil.copytree(TOY_SLICE, slice_dir)
    if not _slice_q.is_file():
        raise FileNotFoundError(
            "webqa_questions.jsonl 없음: "
            + str(_slice_q)
            + "\n→ **0단계** 복사 셀을 실행하거나 data 경로를 확인하세요."
        )
    print("slice 준비 완료:", slice_dir)

_nq = int(N_QUERIES)
for _k in (
    "PATTERN_MAX_SAMPLES",
    "WEBQA_EXPORT_MAX",
    "EXTRACTION_MAX_QUESTIONS",
    "CONSTRUCT_MAX_QUESTIONS",
):
    common_env[_k] = str(_nq)

print("=== 패턴 단계 직전 설정 ===")
print("N_QUERIES (처리 상한):", _nq)
print("RUN_ID / 결과:", globals().get("RUN_ID"))
print("result_dir:", result_dir.resolve())
print("slice:", slice_dir.resolve())
print(
    "  JSONL 존재:",
    _slice_q.is_file(),
    "| 슬라이스 질문 줄 수:",
    sum(1 for _ in _slice_q.open(encoding="utf-8")) if _slice_q.is_file() else "(없음)",
)
print(
    "common_env →",
    "PATTERN_MAX_SAMPLES=%s" % common_env.get("PATTERN_MAX_SAMPLES"),
    "| WEBQA_EXPORT_MAX=%s" % common_env.get("WEBQA_EXPORT_MAX"),
    "| EXTRACTION_MAX_QUESTIONS=%s" % common_env.get("EXTRACTION_MAX_QUESTIONS"),
    "| CONSTRUCT_MAX_QUESTIONS=%s" % common_env.get("CONSTRUCT_MAX_QUESTIONS"),
)



In [ ]:
# pattern.py — 2단계 (바로 위 **2단계 전 점검·쿼리 상한** 셀을 먼저 실행했는지 확인)
import sys

if not all(n in globals() for n in ("REPO", "common_env", "result_dir", "run_step", "_LOCAL_DATA")):
    raise RuntimeError("먼저 위의 **2단계 전 점검·쿼리 상한** 셀과 **준비** 셀을 실행하세요.")

pattern_cache = result_dir / "phase2_pattern_cache"
_local_pattern = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice" / "webqa_questions.jsonl"
pattern_question_file = str(_local_pattern) if _local_pattern.is_file() else str(REPO / "data" / "webqa" / "WebQA_data_first_release" / "WebQA_train_val.json")

pattern_env = {
    **common_env,
    "PATTERN_JSON_FILE_PATH": pattern_question_file,
    "PATTERN_CACHE_DIR": str(pattern_cache),
}
_rc_p2 = run_step("2단계 패턴", [sys.executable, "pattern.py"], pattern_env)
print("패턴 캐시 디렉터리:", pattern_cache, "| 존재:", pattern_cache.is_dir())
if _rc_p2 != 0:
    print("경고: 2단계가 비정상 종료였습니다. 종료 코드:", _rc_p2)




### 패턴 캐시 내용 보기

`phase2_pattern_cache/*.json`에는 질문별 LLM 패턴 응답(`response`)과 원본 질문(`question`)이 저장됩니다. 아래 셀에서 파일 목록과 요약을 출력합니다.


In [ ]:
# phase2_pattern_cache/*.json을 읽어 질문·응답 문자열 앞부분만 미리 봅니다.

import json

def _pattern_entry_parts(data: dict) -> tuple[str, str, str]:
    q = data.get("question") or {}
    if "qid" in q:
        qid, qtext = q["qid"], q.get("question", "")
    else:
        qid, qtext = q.get("Guid", ""), q.get("Q", "")
    resp = data.get("response", "")
    return str(qid), str(qtext).replace("\n", " "), str(resp).replace("\n", " ")


def _short(s: str, n: int) -> str:
    s = s.strip()
    return s if len(s) <= n else s[: n - 3] + "..."


if not pattern_cache.is_dir():
    print("(디렉터리 없음)", pattern_cache)
else:
    files = sorted(pattern_cache.glob("*.json"))
    print(f"JSON 파일 수: {len(files)}  |  경로: {pattern_cache}\n")
    for fp in files:
        with fp.open(encoding="utf-8") as f:
            row = json.load(f)
        qid, qtxt, resp = _pattern_entry_parts(row)
        print("---", fp.name)
        print("  qid:", qid)
        print("  question:", _short(qtxt, 200))
        print("  response:", _short(resp, 280))


### 추출 (`extraction.py`)

**역할:** Phase 2 패턴은 “어떤 종류의 노드·관계를 만들지”만 정하고, **추출** 단계는 실제 `webqa_slice` **텍스트**(및 슬라이스에 연결된 스니펫)를 읽어, 그 안에서 **구체적인 엔티티 이름·설명·관계**를 문자열로 채웁니다. 이 출력은 다음 **4단계 `construct.py`**가 **GraphML**을 만들 때의 재료가 됩니다.

**입력(Input) — 무엇을 읽나**

- **질문 JSONL** (`EXTRACTION_QUESTION_FILE`): 슬라이스의 `webqa_questions.jsonl` 등. 질문별 `Guid`, `Q`, 그리고 `metadata.text_doc_ids`(어떤 텍스트 조각과 묶였는지)를 씁니다.
- **텍스트 JSONL** (`EXTRACTION_TEXT_FILE`): `webqa_texts.jsonl` 등 — 각 줄은 `id`와 `text` 형태로, 위 `text_doc_ids`와 매칭됩니다.
- **패턴 캐시 디렉터리** (`EXTRACTION_PATTERN_CACHE_DIR`): Phase 2의 `phase2_pattern_cache/<Guid>.json`. 파일 안 **`response`** 문자열을 파싱해 `EXTRACT_RELATION_PROMPT`의 **`{Graph_pattern}`** 칸에 들어가는 **타입·관계 안내 블록**을 만듭니다.
- **프롬프트 안의 플레이스홀더:** `prompt.EXTRACT_RELATION_PROMPT`에 있는 **`{input_text}`** 자리에는, 해당 질문에 붙은 스니펫들에서 이어 붙인 **짧은 원문**(코드 상으로는 길이를 잘라 쓸 수 있음)이 들어갑니다.

**출력(Output) — 무엇이 생기나**

- **`phase3_extraction_cache/`** 아래 **`{Guid}_{snippet_id}.json`** 같은 이름의 파일들(예: `_txt` 조각별로 한 파일).
- 각 JSON에는 대략 **`response`**, **`qid`**, **`text_content`**, **`graph_pattern`**, **`llm`** 필드가 있고, **`response`** 안에는 `"entity"<|>...` / `"relationship"<|>...` 형식의 레코드가 **`##`** 로 이어져 있습니다(`EXTRACT_RELATION_PROMPT`의 출력 규격).
- 같은 경로에 이미 같은 이름의 파일이 있으면 **`extraction.py`는 건너뛰어** 재실행 비용을 줄입니다 — 설정을 바꿨으면 해당 캐시를 지우거나 다른 `EXTRACTION_CACHE_DIR`을 쓰는 편이 안전합니다.

**왜 이 단계가 필요한가**

- 패턴만으로는 노드 문자열과 근거가 없고, **텍스트에서 실체를 끌어낸 뒤**에야 비로소 **그래프를 조립하고(construct), 검색·추론(inference)**에 쓸 **지역 지식 그래프**가 성립합니다.

실행 순서: 아래 **프롬프트 출력** 코드 셀에서 `EXTRACT_RELATION_PROMPT` 전체를 확인한 다음, **`extraction.py` 실행** 셀을 실행합니다.


In [ ]:
# extraction.py가 사용하는 프롬프트 템플릿: `prompt.EXTRACT_RELATION_PROMPT`
# 실행 시 `{Graph_pattern}` · `{input_text}` 는 `extraction.py` 본문에서 실제 패턴·텍스트로 치환됩니다.

from prompt import EXTRACT_RELATION_PROMPT

print("EXTRACT_RELATION_PROMPT — extraction.py의 LLM 입력 템플릿\n")
print(EXTRACT_RELATION_PROMPT)



In [ ]:
# extraction.py가 패턴 캐시와 문맥 JSONL을 써서 엔티티·관계 후보를 phase3에 저장합니다.

extraction_cache = result_dir / "phase3_extraction_cache"
extraction_env = {
    **common_env,
    "EXTRACTION_QUESTION_FILE": str(q_file),
    "EXTRACTION_TEXT_FILE": str(t_file),
    "EXTRACTION_PATTERN_CACHE_DIR": str(pattern_cache),
    "EXTRACTION_CACHE_DIR": str(extraction_cache),
}
_rc_p3 = run_step("3단계 추출", [sys.executable, "extraction.py"], extraction_env)
print("추출 캐시 디렉터리:", extraction_cache, "| 존재:", extraction_cache.is_dir())
if _rc_p3 != 0:
    print("경고: 3단계가 비정상 종료였습니다. 종료 코드:", _rc_p3)

In [ ]:
# 3단계 추출 결과 요약 (`phase3_extraction_cache/`) — 경로·파일 목록·샘플

import json
from pathlib import Path

if "extraction_cache" not in globals():
    raise RuntimeError("먼저 위의 `extraction.py` 실행 셀을 실행해 extraction_cache 가 정의되게 하세요.")
if "REPO" not in globals():
    raise RuntimeError("`REPO` 셀을 먼저 실행하세요.")

_cache = Path(extraction_cache).resolve()
_repo = Path(REPO).resolve()

print("=== Phase 3 extraction 산출물 ===")
print("캐시 디렉터리 (절대):", _cache)
try:
    print("저장소 기준 상대:", _cache.relative_to(_repo))
except ValueError:
    print("(REPO 바깥 경로입니다)")

if not _cache.is_dir():
    print("(아직 디렉터리 없음 — extraction.py가 성공했는지 확인)")
else:
    files = sorted(_cache.glob("*.json"))
    print("JSON 개수:", len(files))
    for fp in files[:25]:
        try:
            rel = fp.resolve().relative_to(_repo)
        except ValueError:
            rel = fp
        print(" ", rel)
    if len(files) > 25:
        print("  ... 이후", len(files) - 25, "개")

    if files:
        sample = files[0]
        with sample.open(encoding="utf-8") as fh:
            row = json.load(fh)
        resp = row.get("response") or ""
        resp_s = resp.replace(chr(10), " ").strip()
        if len(resp_s) > 240:
            resp_s = resp_s[:237] + "..."
        print("\n샘플(첫 번째 파일):", sample.name)
        print("  qid:", row.get("qid"))
        print("  text_content 길이:", len(str(row.get("text_content") or "")))
        print("  response 앞부분:", resp_s)



### GraphML 구성 (`construct.py`)

**역할:** 3단계 `phase3_extraction_cache/`의 **`response` 문자열**을 파싱해 **엔티티·관계 노드·엣지**로 만들고, 슬라이스의 **이미지·(선택) 테이블** 메타데이터를 붙여 질문마다 하나의 **NetworkX 무방향 그래프**를 만든 뒤 **`GraphML`(XML)** 로 내보냅니다. 이 출력이 이후 검색·추론(`inference`) 단계의 **구조화된 근거**가 됩니다.

**입력(Input) — 노트북에서 넘기는 값**

| 환경 변수 (노트북 `construct_env`) | 역할 |
|--------------------------------------|------|
| `CONSTRUCT_EXTRACTION_CACHE` | `extraction_cache`와 동일 — `{Guid}_{텍스트스니펫id}.json` 조각을 읽습니다. |
| `CONSTRUCT_QUESTION_FILE` | 슬라이스 질문 JSONL (`Guid`, `metadata.text_doc_ids`, `metadata.image_doc_ids`, `metadata.table_id` 등). |
| `CONSTRUCT_TEXT_FILE` / `IMAGE_FILE` / `TABLE_FILE` | 슬라이스의 텍스트·이미지·테이블 JSONL — 캐시에는 없지만 그래프에 **TABLE/IMAGE 노드**, 캡션·표 마크다운을 채우는 데 씁니다. |
| `CONSTRUCT_WEBQA_SLICE_DIR`(또는 스크립트 기본값 `CONSTRUCT_SLICE_DIR`) | 슬라이스 루트; 위 파일들과 동일 실행 폴더를 가리켜야 합니다. |
| `CONSTRUCT_OUTPUT_GRAPH_DIR` | 여기 노트북은 `result_dir/phase4_graphs_out`(스크립트 기본값은 `phase4_graphs_real`이므로 **반드시 env로 고정**). |

**출력(Output)**

- **`<출력>/<Guid>_graph.graphml`** — 노드 속성에 `entity_name`, `type`, `description`, `source_id`, `doc_id` 등이 들어갑니다.
- **`<출력>/<Guid>_graph.models.json`** — GraphML 표준에 억지로 넣기 어려운 **Phase 3 추출 시 사용한 LLM 정보** 등을 같은 stem으로 기록합니다.

**왜 이 단계가 필요한가**

- 추출 결과는 줄 단위 문자열이라 그대로는 경로 검색이나 검색 서브그래프를 만들기 어렵고, **정식 그래프 파일**로 굳혀야 시각화·ColEmbed 검색 단계와 맞물립니다.

아래 코드 셀이 `construct_env`로 경로를 묶습니다. **3단계 추출 직후** 같은 `RUN_ID` / `result_dir`에서 실행해야 캐시와 맞습니다.


In [ ]:
# 4단계 construct.py — Phase 3 캐시 + 슬라이스 → 질문별 GraphML (+ .models.json)
# CONSTRUCT_EXTRACTION_CACHE: 추출 JSON 위치 (= extraction_cache 와 동일 경로 유지 권장)
# CONSTRUCT_OUTPUT_GRAPH_DIR: 이 노트북은 phase4_graphs_out 사용 (코드 기본 phase4_graphs_real 과 다름)

graph_dir = result_dir / "phase4_graphs_out"
construct_env = {
    **common_env,
    "CONSTRUCT_QUESTION_FILE": str(q_file),
    "CONSTRUCT_TABLE_FILE": str(slice_dir / "webqa_tables.jsonl"),
    "CONSTRUCT_IMAGE_FILE": str(slice_dir / "webqa_images.jsonl"),
    "CONSTRUCT_TEXT_FILE": str(t_file),
    "CONSTRUCT_EXTRACTION_CACHE": str(extraction_cache),
    "CONSTRUCT_OUTPUT_GRAPH_DIR": str(graph_dir),
    "CONSTRUCT_WEBQA_SLICE_DIR": str(slice_dir),
}
_rc_p4 = run_step("4단계 그래프 구성", [sys.executable, "construct.py"], construct_env)
_gml_list = list(graph_dir.glob("*.graphml")) if graph_dir.is_dir() else []
print("GraphML 출력 디렉터리:", graph_dir.resolve(), "| 파일 개수:", len(_gml_list))
if _rc_p4 != 0:
    print("경고: 4단계가 비정상 종료였습니다. 종료 코드:", _rc_p4)


### 추론 (`inference.py`)

켜져 있으면 ColEmbed 검색을 쓰고, 답은 Gemma로 생성합니다(`DRY_RUN`이면 생략).

In [ ]:
# inference.py로 GraphML·슬라이스·ColEmbed 검색을 묶어 답을 생성합니다.
# 이어지는 코드에서 predictions.json을 읽어 예시 몇 건만 출력합니다.

phase5_dir = result_dir / "phase5_inference"
phase5_dir.mkdir(parents=True, exist_ok=True)
pred_json = phase5_dir / "predictions.json"

inference_env = {
    **common_env,
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "INFERENCE_GRAPH_DIR": str(graph_dir),
    "INFERENCE_QUESTION_FILE": str(q_file),
    "INFERENCE_OUTPUT_JSON": str(pred_json),
    "INFERENCE_MAX_QUESTIONS": str(N_QUERIES),
    "INFERENCE_DRY_RUN": dry_run,
    "INFERENCE_COLEMBED_RETRIEVAL": os.environ.get("INFERENCE_COLEMBED_RETRIEVAL", "1"),
    "INFERENCE_WEBQA_SLICE_DIR": str(slice_dir),
    "WEBQA_IMGS_DIR": SHARD14_IMGS,
}
_rc_p5 = run_step("5단계 추론", [sys.executable, "inference.py"], inference_env)

# predictions.json은 qid -> 답 문자열 맵(JSON 객체)
if pred_json.is_file():
    import json as _json

    with pred_json.open(encoding="utf-8") as _f:
        _preds = _json.load(_f)
    _keys = list(_preds.keys()) if isinstance(_preds, dict) else []
    print("\n--- 추론 결과 요약 ---")
    print("출력 파일:", pred_json)
    print("예측 qid 수:", len(_keys))
    for _qid in _keys[:5]:
        _ans = str(_preds[_qid])[:240]
        print(f"  qid={_qid}  답(앞 240자): {_ans!r}")
    if len(_keys) > 5:
        print(f"  ... 외 {len(_keys) - 5}개")
else:
    print("추론 출력 없음:", pred_json)
if _rc_p5 != 0:
    print("경고: 5단계가 비정상 종료였습니다. 종료 코드:", _rc_p5)

### QA 평가 (`eval/evaluate_webqa_qa.py`)

`evaluate_webqa_qa.py`는 `predictions.json`과 슬라이스 **`webqa_questions.jsonl`** 로 QA 지표를 계산하고, **`retrieval_rankings.json`** 이 있으면 검색 블록(hit/recall/MRR 등)도 채웁니다.

Gold JSONL 질문 식별자는 **`Guid`(WebQA 슬라이스)** 와 **`qid`(MMQA 형식)** 모두 허용됩니다(`eval/evaluate_retrieval.py`). 예측·랭킹 JSON의 키는 추론 단계와 같이 **`Guid` 문자열**이면 됩니다.


In [ ]:
# predictions·gold JSONL로 WebQA QA 지표를 계산(evaluate_webqa_qa.py)하고, 생성된 evaluation_report.json을 읽어 요약을 한국어로 출력합니다.

import json
from pathlib import Path


def _print_webqa_report_summary_kr(report_path: Path) -> None:
    """evaluation_report.json에서 리더보드·원시 지표·검색 요약을 한글로 출력."""

    def _f(x) -> float:
        try:
            if x is None:
                return 0.0
            return float(x)
        except (TypeError, ValueError):
            return 0.0

    if not report_path.is_file():
        print("평가 리포트 파일 없음:", report_path)
        return
    with report_path.open(encoding="utf-8") as f:
        rep = json.load(f)
    counts = rep.get("counts") or {}
    diag = rep.get("diagnostics") or {}

    print("\n" + "=" * 62)
    print("WebQA QA 평가 점수 요약")
    print("=" * 62)
    print(
        "샘플 수 — 전체:",
        counts.get("All"),
        "| 채점된 qid:",
        counts.get("scored"),
        "| 단일모달:",
        counts.get("Unimodal"),
        "| 멀티모달:",
        counts.get("Multimodal"),
    )
    if diag.get("webqa_fluency_effective_backend"):
        print(
            "유창도 백엔드:",
            diag.get("webqa_fluency_effective_backend"),
            "| 모델:",
            diag.get("webqa_fluency_model", ""),
        )

    lb = rep.get("leaderboard_summary") or {}
    labels = {"All": "전체"}
    for bucket in ("All",):
        b = lb.get(bucket) or {}
        if not b:
            continue
        print(
            f"  [{labels[bucket]}]  QA-FL={_f(b.get('QA-FL')):.4f}  "
            f"QA-Acc={_f(b.get('QA-Acc')):.4f}  QA={_f(b.get('QA')):.4f}"
        )

    scores = rep.get("scores") or {}
    all_s = scores.get("All") or {}
    if all_s:
        print(
            "원시 list F1 / list EM (전체):",
            f"{_f(all_s.get('f1')):.4f}",
            "/",
            f"{_f(all_s.get('em')):.4f}",
        )
        print(
            "키워드 list F1 / list EM (전체):",
            f"{_f(all_s.get('list_f1_keyword')):.4f}",
            "/",
            f"{_f(all_s.get('list_em_keyword')):.4f}",
        )
        if all_s.get("webqa_acc_approx") is not None:
            print(f"WebQA acc 근사 (전체): {_f(all_s.get('webqa_acc_approx')):.4f}")

    retr = rep.get("retrieval_summary") or {}
    if retr.get("status") == "missing_rankings_json":
        print("검색 지표: retrieval_rankings.json 없음 (해당 시 result/<RUN_ID>/에 배치)")
    elif retr and not retr.get("status"):
        k = int(retr.get("k_gen", 10))
        overall = retr.get("overall") or {}
        metrics_block = overall.get("metrics") or {}
        m = metrics_block.get(f"k={k}")
        if m:
            print(
                f"검색@{k} (전체)  hit@k={_f(m.get('hit@k')):.4f}  "
                f"recall@k={_f(m.get('recall@k')):.4f}  "
                f"MRR={_f(m.get('mrr@k')):.4f}  "
                f"retrieval F1={_f(m.get('retrieval_f1')):.4f}"
            )

    by_acc = scores.get("by_Qcate_webqa_acc_approx") or {}
    if by_acc:
        parts = [f"{c}={_f(v):.3f}" for c, v in sorted(by_acc.items())]
        print("Qcate별 WebQA acc 근사:", " | ".join(parts))
    by_qa = scores.get("by_Qcate_webqa_qa") or {}
    if by_qa:
        parts = [f"{c}={_f(v):.3f}" for c, v in sorted(by_qa.items())]
        print("Qcate별 QA(리더보드):", " | ".join(parts))

    print("상세 JSON:", report_path.resolve())
    print("=" * 62 + "\n")


if pred_json.is_file():
    report_json = phase5_dir / "evaluation_report.json"
    eval_env = {**common_env, "MMGRAPHRAG_RUN_ID": RUN_ID}
    _rc_p6 = run_step(
        "6단계 평가",
        [
            sys.executable,
            "eval/evaluate_webqa_qa.py",
            "--predictions",
            str(pred_json),
            "--gold_jsonl",
            str(q_file),
            "--report_json",
            str(report_json),
            "--split_label",
            "val",
        ],
        eval_env,
    )
    if _rc_p6 == 0:
        _print_webqa_report_summary_kr(report_json)
    else:
        print("평가 스크립트 비정상 종료 — 위쪽 영문 로그를 확인하세요. 종료 코드:", _rc_p6)
else:
    print("평가 건너뜀: predictions.json 없음")


## 부록: 터미널용 원시 셸 명령(선택)

노트북에서는 **단계별 실행(수동)** 셀이 권장입니다. 아래는 터미널만 쓸 때의 **bash** 예시입니다. `$RUN_ID`, 경로, 제한 값은 환경에 맞게 바꾸세요.

**0단계 — 슬라이스 내보내기** (`webqa_slice`를 이미 `result/$RUN_ID/`에 복사했다면 생략):
```bash
export MMGRAPHRAG_RUN_ID=my_run WEBQA_EXPORT_MAX=5 WEBQA_SLICE_DIR=result/my_run/webqa_slice
python export_webqa_slice.py
```

**2단계 — 패턴:**
```bash
export PATTERN_MAX_SAMPLES=5 PATTERN_CACHE_DIR=result/my_run/phase2_pattern_cache
python pattern.py
```

**3단계 — 추출:**
```bash
export EXTRACTION_QUESTION_FILE=result/my_run/webqa_slice/webqa_questions.jsonl ...
python extraction.py
```

환경 변수 배선은 `tests/test_pipeline.py`와 동일합니다 — 재현하려면 거기서 복사하면 됩니다.

## 데모 UI(선택)

`result/<RUN_ID>/`에 예측과 그래프가 있으면 `demo/be/config/paths.yaml`을 설정합니다(보통 `run_id: latest`). 자세한 내용은 `demo/README.md`.

**터미널 명령**이나 아래 **데모 BE+FE 시작** 셀로 노트북에서 서버를 띄울 수 있습니다(백그라운드 `subprocess`). 필요 조건:

- **BE:** 파이프라인과 같은 Python venv; 포트 **8000**(환경 변수 `DEMO_BE_PORT`로 변경).
- **FE:** **Node.js + npm**(예: nvm LTS); 포트 **5173**(`DEMO_FE_PORT`). Vite가 `/api`를 백엔드로 프록시합니다.

브라우저: **http://127.0.0.1:5173/** (FE) — API 상태: **http://127.0.0.1:8000/health**.

### 터미널에서 동일하게

```bash
cd demo/be && python server.py --host 0.0.0.0 --port 8000
cd demo/fe && npm install && npm run dev
```

### 노트북에서 데모 BE + FE 시작

서버를 **백그라운드**로 띄워 노트북을 계속 쓸 수 있습니다. 먼저 **저장소 루트와 경로** 셀을 실행해 `REPO`가 정의되게 하세요(없으면 현재 작업 디렉터리로 추정).

이 셀을 다시 실행하면 이 세션에서 이전에 띄운 서버(`DEMO_PROCESSES`)를 먼저 종료합니다.

In [ ]:
# demo/be server.py와 demo/fe npm run dev를 백그라운드로 띄워 브라우저에서 결과를 볼 수 있게 합니다.

import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "demo" / "be" / "server.py").is_file() else _here.parent

be_dir = _repo / "demo" / "be"
fe_dir = _repo / "demo" / "fe"
if not (be_dir / "server.py").is_file():
    raise FileNotFoundError(f"데모 백엔드를 찾을 수 없음: {be_dir}")

if "DEMO_PROCESSES" not in globals():
    DEMO_PROCESSES = []
else:
    for _p in list(DEMO_PROCESSES):
        if _p.poll() is None:
            _p.terminate()
    DEMO_PROCESSES.clear()

BE_PORT = int(os.environ.get("DEMO_BE_PORT", "8000"))
FE_PORT = int(os.environ.get("DEMO_FE_PORT", "5173"))

be_env = {**os.environ, "PYTHONPATH": str(be_dir)}
p_be = subprocess.Popen(
    [sys.executable, "server.py", "--host", "0.0.0.0", "--port", str(BE_PORT)],
    cwd=str(be_dir),
    env=be_env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
DEMO_PROCESSES.append(p_be)
print(f"데모 BE 시작 | pid={p_be.pid} | http://127.0.0.1:{BE_PORT}/health")
time.sleep(1.5)

npm = shutil.which("npm")
if npm is None:
    print("PATH에 npm 없음 — Node.js 설치 후: cd demo/fe && npm install && npm run dev")
else:
    if not (fe_dir / "node_modules").is_dir():
        print("demo/fe에서 npm install 실행 중(처음은 1분 정도 걸릴 수 있음)...")
        subprocess.run([npm, "install"], cwd=str(fe_dir), check=False)
    p_fe = subprocess.Popen(
        [npm, "run", "dev"],
        cwd=str(fe_dir),
        env={**os.environ},
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    DEMO_PROCESSES.append(p_fe)
    print(f"데모 FE 시작 | pid={p_fe.pid} | http://127.0.0.1:{FE_PORT}/")

print("\n위 프론트 URL을 브라우저에서 엽니다. 종료는 다음 **데모 서버 중지** 셀을 실행하세요.")

### 데모 서버 중지

**데모 BE + FE 시작**에서 기록한 `DEMO_PROCESSES`를 종료합니다. 8000/5173이 여전히 열려 있으면 터미널에서 직접 중지하세요(`kill`, Windows는 작업 관리자).

In [ ]:
# 위에서 띄운 데모 BE/FE 자식 프로세스를 종료합니다.

import time as _time

if "DEMO_PROCESSES" in globals() and DEMO_PROCESSES:
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.terminate()
    _time.sleep(0.5)
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.kill()
    DEMO_PROCESSES.clear()
    print("데모 BE/FE 프로세스를 종료했습니다.")
else:
    print("중지할 DEMO_PROCESSES 없음(먼저 데모 BE + FE 시작을 실행하세요).")

## 더 읽을 거리

- `README.md` — 개요, 데이터 구조, 모델 다운로드, 파이프라인 요약.
- `tests/test_pipeline.py` — 전체 오케스트레이션과 단계별 환경 변수.